# Model 1 v3 · Notebook 1 · Teachers

Model 1 v3 (repo `PLAN.md` §10) distils a 21M meeting-notes student from two teachers. This
notebook runs them **once** and exports only their per-sentence outputs; Notebook 2 trains the
student from those files and never loads a teacher.

1. **Qwen3-0.6B**, the smallest Qwen3 (vLLM, its own venv, before JAX starts). Per meeting window: a capped *thinking*
   pass, then a greedy JSON answer `[[id, score 0-3, "reason", depends_on], ...]` with token
   log-probabilities → a **soft score** per sentence, a **reason** and the sentence it
   **depends on**. One shard per GPU; resumable.
2. **Quality gate** on validation: Qwen must beat both the keyword rule and the best Lead-k on
   importance F1 against the gold labels, or Notebook 2 will not distil it.
3. **RoBERTa-Large** (teacher) and **DistilBERT** (baseline row) fine-tuned once on the gold
   importance labels with KerasHub on JAX; per-sentence logits for every split.

**Inputs:** the `meeting-notes-extractive-v1` dataset (required, v1.1); the KerasHub models
`roberta_large_en` and `distil_bert_base_en_uncased` (Add Input → Models → Keras; required:
a non-interactive session cannot attach models while it runs); a previous output of this
notebook or the merged Qwen tables (optional: Qwen is then not run again). **Accelerator:** GPU T4 x2, Internet on.
**Outputs** (`/kaggle/working`): `data/labels/model1_v3/{qwen,roberta,distilbert}/<split>.parquet`,
`qwen/gate.json`, Qwen's raw thoughts and answers (`qwen/shards/*/raw_<split>.jsonl`) and
`manifest_model1_teachers.json`.

Smoke mode (`SMOKE=1`, CPU, used by the repo's tests): a tiny corpus, a rule-based fake Qwen and
a toy encoder; nothing is downloaded.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # before any keras import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import time
from pathlib import Path

SESSION_START = time.time()
ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE = os.environ.get("SMOKE") == "1"  # tiny CPU run: fake Qwen, toy encoders (< 2 min)
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
INPUT = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
STAGE = "model1_teachers"
print(f"on_kaggle={ON_KAGGLE} smoke={SMOKE} work={WORK}")

In [ ]:
# Exact versions for the encoder environment (Qwen gets its own venv below).
import subprocess

if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.10.2",
                    "keras==3.15.1", "tokenizers==0.23.3", "pyarrow"], check=True)
    try:
        import keras_hub  # noqa: F401  (preinstalled on Kaggle)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras-hub"], check=True)

## Configuration
The same values as the repo's `model1_v3.yaml` (`teacher`, `encoders`, `baselines`).

In [ ]:
CONFIG = {
    "seed": 0,
    "max_hours": 11,                      # per Kaggle session (12 h cap)
    "qwen_share": 0.6,                    # Qwen stops at 60% of the session (resumes next time)
    "teacher": {
        "model": "Qwen/Qwen3-0.6B",       # the smallest Qwen3 (4B ran at ~40 tok/s on T4)
        "fallback_model": "Qwen/Qwen3-1.7B",  # set "model" to this if 0.6B fails the gate
        "vllm_spec": "vllm==0.31.0",
        "python": "3.12",
        "splits": ["validation", "test", "train"],  # validation first: the gate needs it
        "max_train_meetings": 400,        # synthetic train subset
        "max_ami_train_meetings": 30,     # AMI train subset (~16 windows per meeting)
        "window_sentences": 64,
        "window_overlap": 16,
        "batch_meetings": 64,
        "think_temperature": 0.6,
        "top_p": 0.95,
        "max_thinking_tokens": 384,
        "max_answer_tokens": 1280,
        "top_logprobs": 5,
        "max_model_len": 6144,
        "gpu_memory_utilization": 0.9,
        "gate_split": "validation",
        # Use the labels already made once validation and test are complete, even if some
        # train meetings are unlabelled (the student uses gold labels there): no rerun of vLLM.
        "accept_partial_train": True,
    },
    "encoders": {
        "roberta": {"preset": "roberta_large_en", "hf_tokenizer": "FacebookAI/roberta-large",
                    "specials": {"bos": "<s>", "sep": "</s>", "pad": "<pad>"}},
        "distilbert": {"preset": "distil_bert_base_en_uncased",
                       "hf_tokenizer": "distilbert/distilbert-base-uncased",
                       "specials": {"bos": "[CLS]", "sep": "[SEP]", "pad": "[PAD]"}},
        "max_tokens": 512,
        "max_sentence_tokens": 64,
        "max_sentences": 48,
        "epochs": 2,
        "batch_size": 8,
        "peak_lr": 1.0e-5,
        "warmup_fraction": 0.1,
        "weight_decay": 0.01,
        "mixed_precision": "mixed_float16",
        "data_parallel": False,           # one T4: NCCL across 2x T4 failed (2026-10)
    },
    "baselines": {
        "lead_fractions": [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5],
        "keywords": {
            "decision": [r"\b(we|we've|we have) (decided|agreed)\b", r"\bdecision\b",
                         r"\blet's go with\b", r"\b(we'll|we will) go (with|for)\b",
                         r"\bagreed\b", r"\bfinal answer\b"],
            "action": [r"\b(\w+ will|i'll|you'll|he'll|she'll|they'll|we'll) (send|do|take|check|write|prepare|follow|set|book|draft|update|look|fix|call|share|finish)\b",
                       r"\baction item\b",
                       r"\bby (monday|tuesday|wednesday|thursday|friday|tomorrow|end of (the )?(day|week|sprint))\b",
                       r"\bcan you (send|check|take|prepare|update|look)\b"],
        },
    },
}
if SMOKE:
    CONFIG["max_hours"] = 0.2
    CONFIG["teacher"].update(window_sentences=8, window_overlap=2, batch_meetings=2,
                             max_train_meetings=2, max_ami_train_meetings=1)
    CONFIG["encoders"].update(max_tokens=32, max_sentence_tokens=8, max_sentences=6, epochs=1,
                              batch_size=4, mixed_precision="float32")

In [ ]:
import json
import math
import random
import re
import shutil
import subprocess
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from typing import Any

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])


def json_dump(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=str) + "\n")


def hours_left():
    return CONFIG["max_hours"] - (time.time() - SESSION_START) / 3600


def find_dirs(root, predicate):
    """Directories under ``root`` (following links) for which ``predicate(dir)`` holds."""
    found = []
    if root.exists():
        for d, _, _ in os.walk(root, followlinks=True):
            if predicate(Path(d)):
                found.append(Path(d))
    return sorted(found, key=lambda p: len(p.parts))


def find_corpus():
    """The attached corpus release (``data/{synthetic,ami}/sentences/*.parquet``)."""
    found = find_dirs(INPUT, lambda d: (d / "data/synthetic/sentences/train.parquet").exists())
    if not found:
        raise SystemExit("Attach the meeting-notes-extractive-v1 dataset (Add data).")
    return found[0]


LABELS = WORK / "data" / "labels" / "model1_v3"
QWEN = LABELS / "qwen"

# Resume: copy a previous output of this notebook back.
for prev in find_dirs(INPUT, lambda d: (d / f"manifest_{STAGE}.json").exists()
                      or (d / "data/labels/model1_v3").exists()):
    if (prev / "data/labels/model1_v3").exists():
        shutil.copytree(prev / "data", WORK / "data", dirs_exist_ok=True)
        print("restored from", prev)
# Qwen labels from any attached dataset (e.g. results_qwen/): the folder holding
# shards.json + shards/ (the "qwen" folder itself, whatever it is called), wherever it sits;
# copied to data/labels/model1_v3/qwen/.
for qwen_dir in find_dirs(INPUT, lambda d: (d / "shards.json").exists() and (d / "shards").is_dir()):
    shutil.copytree(qwen_dir, QWEN, dirs_exist_ok=True)
    print("Qwen labels restored from", qwen_dir)
# Merged Qwen tables without shards (e.g. salvaged from the raw answers, model1_v3/qwen/):
# gate.json + <split>.parquet are all Notebook 2 needs, so Qwen is not run again.
if not (QWEN / "gate.json").exists():
    for qwen_dir in find_dirs(INPUT, lambda d: d.name == "qwen" and (d / "gate.json").exists()
                              and (d / "validation.parquet").exists()):
        shutil.copytree(qwen_dir, QWEN, dirs_exist_ok=True)
        print("merged Qwen tables restored from", qwen_dir)
        break

**Corpus.** Smoke runs write a tiny corpus in the release layout instead.

In [ ]:
from __future__ import annotations


SPLITS = ("train", "validation", "test")


SOURCES = ("synthetic", "synthetic_hn", "ami")


MEETING_SENTENCES = [
    ("Ana", "Thanks for joining, this is the weekly sync.", 0, "context", False),
    ("Ben", "The load tests failed again on Monday.", 1, "discussion", False),
    ("Ana", "Okay, we hold the release until they pass.", 1, "decision", False),
    ("Ben", "Ben will fix the test rig by Friday.", 1, "action_item", True),
    ("Cleo", "Sounds good.", 0, "discussion", False),
    ("Ana", "See you next week.", 0, "closing", False),
]


def _spans(text: str) -> list[dict[str, Any]]:
    """Owner ("Ben", at the start) and deadline ("Friday") spans of the action-item sentence."""
    day = text.index("Friday")
    return [
        {"start": 0, "end": 3, "role": "owner"},
        {"start": day, "end": day + len("Friday"), "role": "deadline"},
    ]


def tiny_corpus_rows(
    note_id: str, split: str, source: str = "synthetic", repeat: int = 1
) -> list[dict[str, Any]]:
    """Rows of one meeting in the corpus ``sentences`` schema (``repeat`` x 6 sentences)."""
    ami = source == "ami"
    rows = []
    for k in range(repeat):
        for j, (speaker, text, importance, zone, has_spans) in enumerate(MEETING_SENTENCES):
            if ami and zone in ("context", "closing"):
                zone = "discussion"  # AMI has no context / closing annotation
            rows.append(
                {
                    "note_id": note_id,
                    "source": source,
                    "meeting_type": "ami_design" if ami else "standup",
                    "domain": "product_design" if ami else "engineering",
                    "split": split,
                    "sentence_id": k * len(MEETING_SENTENCES) + j,
                    "speaker": speaker,
                    "sentence_text": text,
                    "sentence_text_asr": text.lower().replace(",", "").replace(".", ""),
                    "importance": importance,
                    "confidence": 0.8 if ami else 1.0,
                    "zone": zone,
                    "segment_id": None,
                    "family_id": None,
                    "hard_negative": source == "synthetic_hn",
                    "token_count": len(text.split()),
                    "spans": _spans(text) if has_spans and not ami else [],
                }
            )
    return rows


def tiny_note_row(note_id: str, split: str, rows: list[dict[str, Any]]) -> dict[str, Any]:
    """The ``notes`` row of a meeting (AMI rows carry summary sections)."""
    ami = rows[0]["source"] == "ami"
    return {
        "note_id": note_id,
        "source": rows[0]["source"],
        "meeting_type": rows[0]["meeting_type"],
        "split": split,
        "n_sentences": len(rows),
        "n_words": sum(r["token_count"] for r in rows),
        "n_participants": len({r["speaker"] for r in rows}),
        "summary_abstract": "The team discussed the failing load tests." if ami else None,
        "summary_decisions": "The release is held until the tests pass." if ami else None,
        "summary_actions": "Ben fixes the test rig by Friday." if ami else None,
        "summary_problems": "The load tests fail." if ami else None,
    }


def write_tiny_corpus(corpus_dir: str | Path, meetings_per_split: int = 3) -> None:
    """Write the release layout; meeting ``i`` has ``6 * (1 + i % 6)`` sentences."""
    for split in SPLITS:
        layers: dict[str, tuple[list[dict[str, Any]], list[dict[str, Any]]]] = {
            "synthetic": ([], []),
            "ami": ([], []),
        }
        for i in range(meetings_per_split):
            source = SOURCES[i % 3]
            rows = tiny_corpus_rows(f"{split}-{i}", split, source, repeat=1 + i % 6)
            sentences, notes = layers["ami" if source == "ami" else "synthetic"]
            sentences += rows
            notes.append(tiny_note_row(f"{split}-{i}", split, rows))
        for layer, (sentences, notes) in layers.items():
            for table, records in (("sentences", sentences), ("notes", notes)):
                out = Path(corpus_dir) / "data" / layer / table
                out.mkdir(parents=True, exist_ok=True)
                pq.write_table(pa.Table.from_pylist(records), out / f"{split}.parquet")


if SMOKE:
    CORPUS = WORK / "tiny_corpus"
    write_tiny_corpus(CORPUS, meetings_per_split=6)
else:
    CORPUS = find_corpus()
    # Fail fast: a non-interactive session cannot attach models while it runs, so the KerasHub
    # presets must be attached as inputs before the hours of Qwen labelling start.
    missing = [
        spec["preset"] for name, spec in CONFIG["encoders"].items()
        if isinstance(spec, dict) and "preset" in spec
        and not find_dirs(INPUT, lambda d, p=spec["preset"]: p in d.parts
                          and (d / "config.json").exists())
    ]
    if missing:
        raise SystemExit(f"Attach the KerasHub models {missing} (Add Input -> Models -> Keras).")
print("corpus:", CORPUS)

## 1. Qwen3-0.6B labels (separate environment)
The script below is written to disk and run with the **teacher venv's** Python (vLLM). It is
the repo's `src/model1_notes/teacher/` code: prompt (no example values in the format line),
two-phase generation (capped thinking, then a greedy answer with top-5 log-probs), answer
validation with one greedy retry, soft scores from the score digit's alternatives, windows of
64 sentences (each sentence labelled from the window where it is most central), resumable
JSONL output per shard, merge into Parquet, and the gate.

In [ ]:
TEACHER_SCRIPT = r'''
"""Qwen3-0.6B teacher labelling, merge and quality gate (own environment: vLLM, never JAX/Keras).

Same code as the repo's src/model1_notes/teacher/ (labelling, answers, windows, gate).
"""
import argparse
import json
import math
import random
import re
import sys
import time
import zlib
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

#: Qwen teacher reason classes (PLAN.md 10.3); index = the student's ``reason`` head output.
REASONS = (
    "agreed_outcome",
    "commitment_with_owner",
    "risk_or_blocker",
    "key_fact",
    "status_update",
    "rejected_proposal",
    "tentative",
    "question",
    "filler",
)


def window_starts(length: int, size: int, overlap: int) -> list[int]:
    """Start offsets of windows of ``size`` sentences overlapping by ``overlap``.

    Windows advance by ``size - overlap``; the last one is aligned to the end of the meeting so
    every window except a short meeting's single window is full.
    """
    if not 0 <= overlap < size:
        raise ValueError(f"window_overlap must be in [0, {size}); got {overlap}")
    if length <= size:
        return [0]
    starts = list(range(0, length - size, size - overlap))
    return [*starts, length - size]


def best_window(length: int, size: int, overlap: int) -> list[int]:
    """For each sentence, the window (index into :func:`window_starts`) where it is farthest
    from both window edges (ties go to the earlier window)."""
    starts = window_starts(length, size, overlap)
    best = []
    for i in range(length):
        margins = [min(i - s, s + size - 1 - i) if s <= i < s + size else -1 for s in starts]
        best.append(max(range(len(starts)), key=lambda w: (margins[w], -w)))
    return best


SYSTEM = (
    "You take notes in business meetings. You decide which sentences of a transcript belong in "
    "the notes and why. You answer with JSON only."
)


REASON_RULES = """- "reason": exactly one of
  - "agreed_outcome": something the group explicitly agrees or decides (not a suggestion),
  - "commitment_with_owner": a person commits to a task, usually with a deadline,
  - "risk_or_blocker": a problem, risk or blocker someone raises,
  - "key_fact": an important fact, number or status the notes need,
  - "status_update": routine progress that the notes can skip,
  - "rejected_proposal": a proposal that is turned down, or a decision later reversed,
  - "tentative": an idea, a maybe, a hypothetical, or a task without an owner,
  - "question": a question that gets no answer or commitment,
  - "filler": greetings, thanks, backchannels, small talk, logistics."""


INSTRUCTIONS = """Read the meeting below and decide, for every sentence, whether it belongs in \
the meeting notes.

For each sentence give, in this order:
- the sentence id,
- "score": an integer from 0 to 3:
  3 = a decision, an action item, or a key fact the notes must keep,
  2 = useful detail for the notes,
  1 = context the notes do not need,
  0 = filler, greetings, backchannels, or a bare yes / no / okay,
{reason_rules}
- "depends_on": the id of the earlier sentence that makes this one meaningful (for example the \
proposal an agreement refers to), or -1 if none.

Answer with one JSON array, one entry per sentence, in order, and nothing else:
[[<id>, <score>, "<reason>", <depends_on>], ...]

Meeting ({n} sentences):
{transcript}"""


RETRY = (
    "Your answer was not valid: {error}. Answer again with the JSON array only: exactly one entry "
    '[id, score, "reason", depends_on] per sentence id from 1 to {n}, score an integer 0-3, '
    "reason one of {reasons}, depends_on an earlier id or -1."
)


def transcript(speakers: Sequence[str], sentences: Sequence[str]) -> str:
    """``<id>. <speaker>: <sentence>`` lines, ids from 1."""
    return "\n".join(
        f"{i}. {s}: {t}" for i, (s, t) in enumerate(zip(speakers, sentences, strict=True), 1)
    )


def build_messages(speakers: Sequence[str], sentences: Sequence[str]) -> list[dict[str, str]]:
    """Chat messages (system + user) for one meeting window."""
    user = INSTRUCTIONS.format(
        reason_rules=REASON_RULES, n=len(sentences), transcript=transcript(speakers, sentences)
    )
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def retry_message(error: str, n: int, reasons: Sequence[str]) -> dict[str, str]:
    """User message asking for a corrected answer."""
    names = ", ".join(f'"{r}"' for r in reasons)
    return {"role": "user", "content": RETRY.format(error=error, n=n, reasons=names)}


SCORE_RANGE = (0, 3)


#: ``[ <id> , <score>`` at the start of an entry: group 2 is the score digit.
ENTRY_RE = re.compile(r"\[\s*(\d+)\s*,\s*(-?\d+)")


FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)


#: Chat special tokens such as ``<|im_end|>`` left in the decoded text.
SPECIAL_RE = re.compile(r"<\|[^|>]*\|>")

#: One flat JSON object (an entry in object form).
OBJECT_RE = re.compile(r"\{[^{}]*\}")

#: ``[id, score, "reason", depends_on]`` (an entry in list form).
LIST_RE = re.compile(r'\[\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*"([^"]*)"\s*,\s*(-?\d+)\s*\]')

#: ``"id": <id>, "score": <score>`` in an object entry: group 2 is the score digit.
OBJECT_SCORE_RE = re.compile(r'"id"\s*:\s*(\d+)\s*,\s*"score"\s*:\s*(-?\d+)')


TOO_LONG = "prompt too long for the model context"


class InvalidAnswer(ValueError):
    """The teacher's answer does not follow the format."""


@dataclass
class Label:
    """One sentence's teacher label (``depends_on`` is a 1-based id, -1 = none)."""

    score: int
    reason: str | None  # None: the teacher gave a reason outside the list
    depends_on: int
    soft: float


def _as_int(value: object) -> int | None:
    if isinstance(value, bool):
        return None
    if isinstance(value, int):
        return value
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return None


def answer_body(text: str | None) -> tuple[str, int]:
    """The JSON part of an answer and its character offset in ``text``."""
    if text is None:
        raise InvalidAnswer(TOO_LONG)
    fenced = FENCE_RE.search(text)
    if fenced:
        return fenced.group(1), fenced.start(1)
    start = text.find("[")
    if start < 0:
        raise InvalidAnswer("no JSON array found")
    return text[start:], start


def parse_answer(text: str | None, n: int, reasons: Sequence[str]) -> list[tuple[int, str, int]]:
    """``(score, reason, depends_on)`` for sentence ids 1..n, or :class:`InvalidAnswer`."""
    body, _ = answer_body(text)
    try:
        items, _ = json.JSONDecoder().raw_decode(body)
    except json.JSONDecodeError as err:
        raise InvalidAnswer(f"invalid JSON ({err.msg})") from None
    if not isinstance(items, list):
        raise InvalidAnswer("the answer is not a JSON array")
    lo, hi = SCORE_RANGE
    by_id: dict[int, tuple[int, str, int]] = {}
    for item in items:
        if not isinstance(item, list) or len(item) != 4:
            raise InvalidAnswer('every entry must be [id, score, "reason", depends_on]')
        uid, score, reason, dep = _as_int(item[0]), _as_int(item[1]), item[2], _as_int(item[3])
        if uid is None or not 1 <= uid <= n:
            raise InvalidAnswer(f"id {item[0]!r} is not a sentence number 1..{n}")
        if uid in by_id:
            raise InvalidAnswer(f"id {uid} appears more than once")
        if score is None or not lo <= score <= hi:
            raise InvalidAnswer(f"id {uid}: score {item[1]!r} is not an integer {lo}-{hi}")
        if reason not in reasons:
            raise InvalidAnswer(f"id {uid}: reason {reason!r} is not one of the reasons")
        if dep is None or not (dep == -1 or 1 <= dep < uid):
            raise InvalidAnswer(f"id {uid}: depends_on {item[3]!r} is not -1 or an earlier id")
        by_id[uid] = (score, reason, dep)
    missing = [i for i in range(1, n + 1) if i not in by_id]
    if missing:
        raise InvalidAnswer(f"missing ids {missing[:10]}")
    return [by_id[i] for i in range(1, n + 1)]


def score_positions(text: str) -> dict[int, int]:
    """``id -> character offset`` of each entry's score digit in ``text`` (both formats)."""
    positions = {int(m.group(1)): m.start(2) for m in OBJECT_SCORE_RE.finditer(text)}
    for m in LIST_RE.finditer(text):
        positions.setdefault(int(m.group(1)), m.start(2))
    return positions


def digit_distribution(top: dict[str, float]) -> dict[int, float] | None:
    """Renormalised probabilities of the scores among a token's top log-probabilities."""
    lo, hi = SCORE_RANGE
    probs: dict[int, float] = {}
    for token, logprob in top.items():
        token = token.strip()
        if token.isdigit() and lo <= int(token) <= hi:
            probs[int(token)] = probs.get(int(token), 0.0) + math.exp(logprob)
    total = sum(probs.values())
    return None if total <= 0 else {k: v / total for k, v in probs.items()}


def soft_scores(
    text: str,
    scores: Sequence[int],
    tokens: Sequence[tuple[str, dict[str, float]]] | None,
) -> list[float]:
    """Soft score in [0, 1] per sentence (see module docstring).

    ``tokens`` are the generated tokens as ``(text, {alternative: logprob})``; their texts
    concatenated must equal ``text``.
    """
    hi = SCORE_RANGE[1]
    soft = [s / hi for s in scores]
    if not tokens:
        return soft
    ends, pos = [], 0
    for token_text, _ in tokens:
        pos += len(token_text)
        ends.append(pos)
    for uid, char in score_positions(text).items():
        if not 1 <= uid <= len(scores):
            continue
        k = next((i for i, end in enumerate(ends) if end > char), None)
        dist = None if k is None else digit_distribution(tokens[k][1])
        if dist:
            soft[uid - 1] = sum(score * p for score, p in dist.items()) / hi
    return [round(s, 6) for s in soft]


def parse_entries(
    text: str | None, n: int, reasons: Sequence[str]
) -> dict[int, tuple[int, str | None, int]]:
    """Lenient reading (see module docstring): ``id -> (score, reason, depends_on)``."""
    if not text:
        return {}
    text = SPECIAL_RE.sub("", text)
    lo, hi = SCORE_RANGE
    raw: dict[int, tuple[object, object, object]] = {}
    for m in OBJECT_RE.finditer(text):
        try:
            obj = json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
        uid = _as_int(obj.get("id")) if isinstance(obj, dict) else None
        if uid is not None and uid not in raw:
            raw[uid] = (obj.get("score"), obj.get("reason"), obj.get("depends_on", -1))
    for m in LIST_RE.finditer(text):
        uid = int(m.group(1))
        if uid not in raw:
            raw[uid] = (int(m.group(2)), m.group(3), int(m.group(4)))
    entries: dict[int, tuple[int, str | None, int]] = {}
    for uid, (score, reason, dep) in raw.items():
        score, dep = _as_int(score), _as_int(dep)
        if not 1 <= uid <= n or score is None or not lo <= score <= hi:
            continue
        reason = reason if reason in reasons else None
        dep = dep if dep is not None and 1 <= dep < uid else -1
        entries[uid] = (score, reason, dep)
    return entries


def labels_from_answer(
    text: str | None,
    n: int,
    reasons: Sequence[str],
    tokens: Sequence[tuple[str, dict[str, float]]] | None = None,
    min_coverage: float = 0.0,
) -> dict[int, Label]:
    """Labels with soft scores for the sentences of one window that the answer covers
    (``id -> Label``); raises :class:`InvalidAnswer` below ``min_coverage``."""
    if text is None:
        raise InvalidAnswer(TOO_LONG)
    entries = parse_entries(text, n, reasons)
    if not entries or len(entries) < min_coverage * n:
        raise InvalidAnswer(f"only {len(entries)} of {n} sentences have a valid entry")
    scores = [entries[i][0] if i in entries else 0 for i in range(1, n + 1)]
    soft = soft_scores(text, scores, tokens)
    return {i: Label(s, r, d, soft[i - 1]) for i, (s, r, d) in sorted(entries.items())}


Tokens = list[tuple[str, dict[str, float]]]


@dataclass
class Generation:
    """One window's output: the thought (may be empty), the answer and its token log-probs."""

    thinking: str
    answer: str | None  # None: prompt too long for the context
    tokens: Tokens | None = None


#: ``generate(conversations, think) -> generations``; ``think=False`` answers directly.
Backend = Callable[[list[list[dict[str, str]]], bool], list[Generation]]


def read_meetings(corpus_dir: str | Path, split: str) -> dict[str, list[dict[str, Any]]]:
    """Sentences of every meeting of ``split`` (both layers), sorted by ``sentence_id``."""
    import pyarrow.parquet as pq

    columns = ["note_id", "source", "sentence_id", "speaker", "sentence_text"]
    meetings: dict[str, list[dict[str, Any]]] = {}
    for layer in ("synthetic", "ami"):
        path = Path(corpus_dir) / "data" / layer / "sentences" / f"{split}.parquet"
        if path.exists():
            for row in pq.read_table(path, columns=columns).to_pylist():
                meetings.setdefault(row["note_id"], []).append(row)
    for rows in meetings.values():
        rows.sort(key=lambda r: r["sentence_id"])
    return meetings


def select_meetings(
    meetings: dict[str, list[dict[str, Any]]], split: str, cfg: dict[str, Any], seed: int
) -> list[str]:
    """Meetings to label: all of them, except seeded subsets of the train meetings
    (``max_train_meetings`` synthetic, ``max_ami_train_meetings`` AMI; ``None`` = all)."""
    ids = list(meetings)
    if split != "train":
        return ids
    keep = set()
    for is_ami, key in ((False, "max_train_meetings"), (True, "max_ami_train_meetings")):
        group = [m for m in ids if (meetings[m][0]["source"] == "ami") == is_ami]
        cap = cfg.get(key)
        keep |= set(
            group if cap is None else random.Random(seed).sample(group, min(cap, len(group)))
        )
    return [m for m in ids if m in keep]


def in_shard(note_id: str, shard: int, num_shards: int) -> bool:
    """Stable assignment of a meeting to a shard (one shard per GPU)."""
    return zlib.crc32(note_id.encode()) % num_shards == shard


def label_windows(
    generate: Backend, windows: list[Sequence[dict[str, Any]]], min_coverage: float = 0.9
) -> tuple[list[dict[int, Label] | None], list[dict[str, Any]]]:
    """Labels per window (``id -> Label``, possibly partial; ``None`` = nothing usable) and one
    raw record per window. A window below ``min_coverage`` gets one greedy retry without
    thinking; the better of the two answers is kept."""
    convs = [
        build_messages([r["speaker"] for r in w], [r["sentence_text"] for r in w]) for w in windows
    ]
    first = generate(convs, True)
    labels: list[dict[int, Label] | None] = [None] * len(windows)
    raw: list[dict[str, Any]] = []
    retry, errors = [], {}
    for i, (w, g) in enumerate(zip(windows, first, strict=True)):
        try:
            labels[i] = labels_from_answer(g.answer, len(w), REASONS, g.tokens, min_coverage)
        except InvalidAnswer as err:
            errors[i] = str(err)
            if g.answer is not None and str(err) != TOO_LONG:
                retry.append(i)
            partial = parse_entries(g.answer, len(w), REASONS)
            labels[i] = labels_from_answer(g.answer, len(w), REASONS, g.tokens) if partial else None
        raw.append({"thinking": g.thinking, "answer": g.answer, "error": errors.get(i)})
    if retry:
        convs2 = [
            [
                *convs[i],
                {"role": "assistant", "content": first[i].answer or ""},
                retry_message(errors[i], len(windows[i]), REASONS),
            ]
            for i in retry
        ]
        for i, g in zip(retry, generate(convs2, False), strict=True):
            n = len(windows[i])
            again = (
                labels_from_answer(g.answer, n, REASONS, g.tokens)
                if parse_entries(g.answer, n, REASONS)
                else None
            )
            if again and len(again) > len(labels[i] or {}):
                labels[i] = again
            covered = len(labels[i] or {})
            error = None if covered >= min_coverage * n else f"{errors[i]}; retry: {covered}/{n}"
            raw[i].update(retried=True, retry_answer=g.answer, error=error)
    for r, lab, w in zip(raw, labels, windows, strict=True):
        r["labelled"] = len(lab or {})
        r["sentences"] = len(w)
    return labels, raw


def merge_meeting(
    rows: Sequence[dict[str, Any]],
    starts: Sequence[int],
    window_labels: Sequence[dict[int, Label] | None],
    size: int,
) -> list[dict[str, Any]] | None:
    """Per-sentence labels of a meeting from its (possibly partial) windows.

    Each sentence takes its label from the window that labelled it where it is farthest from
    the window edges; sentences no window labelled are left out. ``None`` if nothing is
    labelled.
    """
    sentences = []
    for i, row in enumerate(rows):
        best, best_margin = None, -1
        for w, (start, labels) in enumerate(zip(starts, window_labels, strict=True)):
            local = i - start + 1
            if labels and start <= i < start + size and local in labels:
                margin = min(i - start, start + size - 1 - i)
                if margin > best_margin:
                    best, best_margin = w, margin
        if best is None:
            continue
        start, label = starts[best], window_labels[best][i - starts[best] + 1]  # type: ignore[index]
        dep = -1 if label.depends_on < 0 else rows[start + label.depends_on - 1]["sentence_id"]
        sentences.append(
            {
                "sentence_id": row["sentence_id"],
                "score": label.soft,
                "raw_score": label.score,
                "reason": label.reason,
                "depends_on": dep,
            }
        )
    return sentences or None


def done_ids(*paths: Path) -> set[str]:
    """Meeting IDs already written to the given JSONL files."""
    ids = set()
    for p in paths:
        if p.exists():
            ids |= {json.loads(line)["note_id"] for line in p.open() if line.strip()}
    return ids


def label_split(
    generate: Backend,
    meetings: dict[str, list[dict[str, Any]]],
    ids: Sequence[str],
    out: Path,
    split: str,
    cfg: dict[str, Any],
    deadline: float,
) -> dict[str, Any]:
    """Label ``ids`` of one split into ``out`` (resumable). Returns this split's counts."""
    size, overlap = cfg["window_sentences"], cfg["window_overlap"]
    good_p, raw_p = out / f"{split}.jsonl", out / f"raw_{split}.jsonl"
    todo = [m for m in ids if m not in done_ids(good_p, raw_p)]
    start, n_good, n_failed = time.time(), 0, 0
    for b in range(0, len(todo), cfg["batch_meetings"]):
        if time.time() > deadline:
            return {"stopped_early": True, "labelled": n_good, "failed": n_failed}
        batch = todo[b : b + cfg["batch_meetings"]]
        starts = {m: window_starts(len(meetings[m]), size, overlap) for m in batch}
        jobs = [(m, s) for m in batch for s in starts[m]]
        windows = [meetings[m][s : s + size] for m, s in jobs]
        labels, raw = label_windows(generate, windows, cfg.get("min_window_coverage", 0.9))
        by_meeting: dict[str, list[dict[int, Label] | None]] = {m: [] for m in batch}
        with raw_p.open("a") as rf:
            for (m, s), lab, r in zip(jobs, labels, raw, strict=True):
                by_meeting[m].append(lab)
                rf.write(json.dumps({"note_id": m, "start": s, **r}) + "\n")
        with good_p.open("a") as gf:
            for m in batch:
                merged = merge_meeting(meetings[m], starts[m], by_meeting[m], size)
                if merged is None:
                    n_failed += 1
                    continue
                n_good += 1
                gf.write(json.dumps({"note_id": m, "sentences": merged}) + "\n")
        print(f"qwen/{split}: {b + len(batch)}/{len(todo)}", flush=True)
    return {
        "stopped_early": False,
        "labelled": n_good,
        "failed": n_failed,
        "seconds": round(time.time() - start, 1),
    }


def fake_backend(seed: int = 0) -> Backend:
    """Smoke runs and tests only: a keyword rule with made-up log-probabilities, never Qwen."""

    def answer_for(conv: list[dict[str, str]]) -> Generation:
        lines = conv[1]["content"].split("sentences):\n", 1)[1].splitlines()
        entries = []
        for i, line in enumerate(lines, 1):
            text = line.split(": ", 1)[1].lower()
            if " will " in f" {text} ":
                score, reason = 3, "commitment_with_owner"
            elif "hold" in text or "decid" in text:
                score, reason = 3, "agreed_outcome"
            elif "failed" in text:
                score, reason = 2, "risk_or_blocker"
            else:
                score, reason = 0, "filler"
            dep = i - 1 if reason == "agreed_outcome" and i > 1 else -1
            entries.append(f'[{i}, {score}, "{reason}", {dep}]')
        pieces = ["[", *(e + ", " for e in entries[:-1]), entries[-1], "]"]
        tokens: Tokens = []
        for piece in pieces:  # the score digit is its own token, with alternatives
            head, sep, rest = piece.partition(", ")
            tokens.append((head + sep, {}))
            if rest:
                digit, rest2 = rest[0], rest[1:]
                alt = {digit: -0.1, str((int(digit) + 1) % 4): -2.5}
                tokens += [(digit, alt), (rest2, {})]
        text = "".join(t for t, _ in tokens)
        return Generation("Thinking about who decided what.", text, tokens)

    def generate(convs: list[list[dict[str, str]]], think: bool) -> list[Generation]:
        return [answer_for(c[:2]) for c in convs]

    return generate


def vllm_backend(cfg: dict[str, Any], seed: int) -> Backend:  # pragma: no cover - Kaggle only
    """Qwen3 through vLLM: budget-forced thinking, then a greedy answer with log-probs."""
    from vllm import LLM, SamplingParams

    llm = LLM(
        model=cfg["model"],
        dtype="half",
        max_model_len=cfg["max_model_len"],
        gpu_memory_utilization=cfg["gpu_memory_utilization"],
        enable_prefix_caching=True,
        seed=seed,
    )
    tok = llm.get_tokenizer()
    think = SamplingParams(
        temperature=cfg["think_temperature"],
        top_p=cfg["top_p"],
        max_tokens=cfg["max_thinking_tokens"],
        stop=["</think>"],
        seed=seed,
    )
    answer = SamplingParams(
        temperature=0.0, max_tokens=cfg["max_answer_tokens"], logprobs=cfg["top_logprobs"]
    )

    def tokens_of(out: Any) -> Tokens:
        tokens = []
        for tid, alts in zip(out.token_ids, out.logprobs or [], strict=False):
            chosen = alts[tid].decoded_token if tid in alts else ""
            tokens.append((chosen, {a.decoded_token: a.logprob for a in alts.values()}))
        return tokens

    def generate(convs: list[list[dict[str, str]]], use_thinking: bool) -> list[Generation]:
        prompts = [
            tok.apply_chat_template(
                c, tokenize=False, add_generation_prompt=True, enable_thinking=use_thinking
            )
            for c in convs
        ]
        budget = cfg["max_answer_tokens"] + (cfg["max_thinking_tokens"] if use_thinking else 0)
        fits = [len(tok.encode(p)) + budget <= cfg["max_model_len"] for p in prompts]
        results = [Generation("", None) for _ in convs]
        ok = [i for i, f in enumerate(fits) if f]
        thoughts = {i: "" for i in ok}
        if use_thinking and ok:
            for i, out in zip(ok, llm.generate([prompts[i] for i in ok], think), strict=True):
                thoughts[i] = out.outputs[0].text
        second = []
        for i in ok:
            if use_thinking:
                thought = thoughts[i]
                if not thought.lstrip().startswith("<think>"):
                    thought = "<think>\n" + thought
                second.append(prompts[i] + thought.rstrip() + "\n</think>\n\n")
            else:
                second.append(prompts[i])
        for i, out in zip(ok, llm.generate(second, answer), strict=True):
            tokens = tokens_of(out.outputs[0])
            text = "".join(t for t, _ in tokens) or out.outputs[0].text
            results[i] = Generation(thoughts[i], text, tokens)
        return results

    return generate


def write_stats(path: Path, stats: dict[str, Any]) -> None:
    path.write_text(json.dumps(stats, indent=2) + "\n")


def run(run_cfg: dict[str, Any], backend: str, deadline: float, shard: int, num_shards: int) -> int:
    """Label every configured split for one shard. Returns 0 (done or stopped at deadline)."""
    cfg, seed = run_cfg["teacher"], run_cfg["seed"]
    out = Path(run_cfg["out_dir"]) / "qwen" / "shards" / str(shard)
    out.mkdir(parents=True, exist_ok=True)
    if backend == "fake":
        generate, info = fake_backend(seed), {"backend": "fake (smoke only)", "model": None}
    else:  # pragma: no cover - Kaggle only
        try:
            import vllm

            generate = vllm_backend(cfg, seed)
            info = {"backend": f"vllm {vllm.__version__}", "model": cfg["model"]}
        except Exception as err:  # not installed, or cannot start on this GPU
            print(f"vLLM unusable: {type(err).__name__}: {err}", flush=True)
            return 3
    stats_path = out / "stats.json"
    stats = json.loads(stats_path.read_text()) if stats_path.exists() else {"splits": {}}
    stats.update(info, complete=False)
    for split in cfg["splits"]:
        meetings = read_meetings(run_cfg["corpus_dir"], split)
        ids = [
            m for m in select_meetings(meetings, split, cfg, seed) if in_shard(m, shard, num_shards)
        ]
        result = label_split(generate, meetings, ids, out, split, cfg, deadline)
        prev = stats["splits"].get(split, {})
        result["seconds"] = round(prev.get("seconds", 0) + result.get("seconds", 0), 1)
        result["meetings"] = len(ids)
        result["labelled"] = len(done_ids(out / f"{split}.jsonl"))
        result["failed"] = len(done_ids(out / f"raw_{split}.jsonl")) - result["labelled"]
        result["failure_rate"] = round(result["failed"] / max(len(ids), 1), 4)
        stats["splits"][split] = result
        write_stats(stats_path, stats)
        if result["stopped_early"]:
            print("deadline reached, stopping (resume next session)", flush=True)
            return 0
    stats["complete"] = True
    write_stats(stats_path, stats)
    return 0


def merge_shards(out_dir: str | Path, num_shards: int, splits: Sequence[str]) -> dict[str, int]:
    """``qwen/shards/*/<split>.jsonl`` -> ``qwen/<split>.parquet`` (one row per sentence)."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    root = Path(out_dir) / "qwen"
    counts = {}
    for split in splits:
        rows = []
        for k in range(num_shards):
            path = root / "shards" / str(k) / f"{split}.jsonl"
            if path.exists():
                for line in path.open():
                    if line.strip():
                        rec = json.loads(line)
                        rows += [{"note_id": rec["note_id"], **s} for s in rec["sentences"]]
        rows.sort(key=lambda r: (r["note_id"], r["sentence_id"]))
        schema = pa.schema(
            [
                ("note_id", pa.string()),
                ("sentence_id", pa.int32()),
                ("score", pa.float32()),
                ("raw_score", pa.int8()),
                ("reason", pa.string()),
                ("depends_on", pa.int32()),
            ]
        )
        pq.write_table(pa.Table.from_pylist(rows, schema=schema), root / f"{split}.parquet")
        counts[split] = len(rows)
    return counts


ZONE_DISCUSSION, ZONE_DECISION, ZONE_ACTION = "discussion", "decision", "action_item"


def lead_fraction(n: int, fraction: float) -> list[int]:
    """1 for the first ``ceil(fraction * n)`` sentences (at least one), else 0."""
    k = max(1, math.ceil(fraction * n - 1e-9))
    return [1 if i < k else 0 for i in range(n)]


class KeywordRule:
    """Decision / action trigger regexes (case-insensitive) from ``cfg``."""

    def __init__(self, cfg: dict[str, Sequence[str]]) -> None:
        self.decision = [re.compile(p, re.IGNORECASE) for p in cfg["decision"]]
        self.action = [re.compile(p, re.IGNORECASE) for p in cfg["action"]]

    def zone(self, text: str) -> str:
        """Predicted zone: action triggers win over decision triggers."""
        if any(p.search(text) for p in self.action):
            return ZONE_ACTION
        if any(p.search(text) for p in self.decision):
            return ZONE_DECISION
        return ZONE_DISCUSSION

    def importance(self, text: str) -> int:
        """1 if the sentence matches any trigger."""
        return int(self.zone(text) != ZONE_DISCUSSION)


def binary_f1(truth: Sequence[int], pred: Sequence[int]) -> float:
    """F1 of the positive class (0 when nothing is predicted or labelled positive)."""
    tp = sum(1 for t, p in zip(truth, pred, strict=True) if t and p)
    denom = sum(truth) + sum(pred)
    return 0.0 if denom == 0 else 2 * tp / denom


def tune_lead(meetings: Sequence[Sequence[int]], grid: Sequence[float]) -> tuple[float, float]:
    """Fraction in ``grid`` with the best pooled importance F1 over ``meetings`` (gold 0/1)."""
    best = (grid[0], -1.0)
    for fraction in grid:
        truth = [t for m in meetings for t in m]
        pred = [p for m in meetings for p in lead_fraction(len(m), fraction)]
        f1 = binary_f1(truth, pred)
        if f1 > best[1]:
            best = (fraction, f1)
    return best


REASON_FOR_ZONE = {"decision": "agreed_outcome", "action_item": "commitment_with_owner"}


def gate_metrics(
    gold: list[dict[str, Any]],
    qwen: dict[tuple[str, int], dict[str, Any]],
    rule: KeywordRule,
    lead_fractions: list[float],
) -> dict[str, Any]:
    """Gate metrics on the gold sentences that Qwen labelled (see module docstring)."""
    rows = [
        (g, qwen[(g["note_id"], g["sentence_id"])])
        for g in gold
        if (g["note_id"], g["sentence_id"]) in qwen
    ]
    truth = [g["importance"] for g, _ in rows]
    pred = [int(q["score"] > 0.5) for _, q in rows]
    keyword = [rule.importance(g["sentence_text"]) for g, _ in rows]
    hn = [i for i, (g, _) in enumerate(rows) if g["hard_negative"] and not g["importance"]]
    agree = [
        q["reason"] == REASON_FOR_ZONE[g["zone"]]
        for g, q in rows
        if g["importance"] and g["zone"] in REASON_FOR_ZONE
    ]
    q_f1, k_f1 = binary_f1(truth, pred), binary_f1(truth, keyword)
    meetings: dict[str, list[tuple[int, int]]] = {}
    for g, _ in rows:
        meetings.setdefault(g["note_id"], []).append((g["sentence_id"], g["importance"]))
    ordered = [[imp for _, imp in sorted(m)] for m in meetings.values()]
    lead_frac, l_f1 = tune_lead(ordered, lead_fractions) if ordered else (None, 0.0)
    return {
        "sentences": len(rows),
        "coverage": round(len(rows) / max(len(gold), 1), 4),
        "importance_f1": round(q_f1, 4),
        "keyword_f1": round(k_f1, 4),
        "lead_f1": round(l_f1, 4),
        "lead_fraction": lead_frac,
        "hard_negative_fp": round(sum(pred[i] for i in hn) / max(len(hn), 1), 4),
        "keyword_hard_negative_fp": round(sum(keyword[i] for i in hn) / max(len(hn), 1), 4),
        "reason_zone_agreement": round(sum(agree) / max(len(agree), 1), 4),
        "passed": bool(rows) and q_f1 > max(k_f1, l_f1),
    }


def run_gate(
    corpus_dir: str | Path, out_dir: str | Path, baselines: dict[str, Any], split: str
) -> dict[str, Any]:
    """Compute the gate on ``split`` and write ``<out_dir>/qwen/gate.json``."""
    import pyarrow.parquet as pq

    columns = ["note_id", "sentence_id", "sentence_text", "importance", "zone", "hard_negative"]
    gold = []
    for layer in ("synthetic", "ami"):
        path = Path(corpus_dir) / "data" / layer / "sentences" / f"{split}.parquet"
        if path.exists():
            gold += pq.read_table(path, columns=columns).to_pylist()
    table = pq.read_table(Path(out_dir) / "qwen" / f"{split}.parquet").to_pylist()
    qwen = {(r["note_id"], r["sentence_id"]): r for r in table}
    rule = KeywordRule(baselines["keywords"])
    result = {"split": split, **gate_metrics(gold, qwen, rule, baselines["lead_fractions"])}
    (Path(out_dir) / "qwen" / "gate.json").write_text(json.dumps(result, indent=2) + "\n")
    return result


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--mode", choices=["label", "merge", "gate"], default="label")
    ap.add_argument("--backend", choices=["vllm", "fake"], default="vllm")
    ap.add_argument("--deadline", type=float, default=float("inf"))
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--num-shards", type=int, default=1)
    args = ap.parse_args()
    run_cfg = json.loads(Path(args.config).read_text())
    if args.mode == "label":
        sys.exit(run(run_cfg, args.backend, args.deadline, args.shard, args.num_shards))
    if args.mode == "merge":
        print(merge_shards(run_cfg["out_dir"], args.num_shards, run_cfg["teacher"]["splits"]))
        return
    print(run_gate(run_cfg["corpus_dir"], run_cfg["out_dir"], run_cfg["baselines"],
                   run_cfg["teacher"]["gate_split"]))


if __name__ == "__main__":
    main()
'''

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def gpu_count():
    """GPUs visible to the driver: nvidia-smi if on PATH, else /proc (0 on CPU)."""
    if shutil.which("nvidia-smi"):
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout
        return len(out.split())
    proc = Path("/proc/driver/nvidia/gpus")
    return len(list(proc.iterdir())) if proc.exists() else 0


def shard_complete(k):
    path = QWEN / "shards" / str(k) / "stats.json"
    return path.exists() and json.loads(path.read_text()).get("complete", False)


script, run_cfg_path = Path("/tmp/teacher_v3.py"), Path("/tmp/teacher_v3.json")
script.write_text(TEACHER_SCRIPT)
run_cfg_path.write_text(json.dumps({"teacher": CONFIG["teacher"], "seed": CONFIG["seed"],
                                    "corpus_dir": str(CORPUS), "out_dir": str(LABELS),
                                    "baselines": CONFIG["baselines"]}))
QWEN_STATUS = "complete" if (QWEN / "gate.json").exists() else "pending"


def eval_splits_done():
    """Every shard finished validation and test (the gate and the evaluation need them)."""
    if not (QWEN / "shards.json").exists():
        return False
    n = json.loads((QWEN / "shards.json").read_text())["num_shards"]
    for k in range(n):
        path = QWEN / "shards" / str(k) / "stats.json"
        splits = json.loads(path.read_text())["splits"] if path.exists() else {}
        for split in ("validation", "test"):
            if split not in splits or splits[split].get("stopped_early"):
                return False
    return True


if QWEN_STATUS == "pending" and CONFIG["teacher"]["accept_partial_train"] and eval_splits_done():
    n_shards = json.loads((QWEN / "shards.json").read_text())["num_shards"]
    print("validation + test labelled on every shard: merging what exists (no new labelling)")
    for mode in ("merge", "gate"):
        subprocess.run([sys.executable, str(script), "--config", str(run_cfg_path),
                        "--mode", mode, "--num-shards", str(n_shards)], check=True)
    QWEN_STATUS = "complete (train partly labelled)"
if QWEN_STATUS == "pending":
    deadline = SESSION_START + CONFIG["qwen_share"] * CONFIG["max_hours"] * 3600
    # The venv is built with uv: Kaggle's Python has no ensurepip (`python -m venv` fails).
    venv = Path("/tmp/teacher_env")
    if SMOKE:
        python, backend = sys.executable, "fake"
    else:
        if not (venv / ".ready").exists():
            shutil.rmtree(venv, ignore_errors=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
            uv = [sys.executable, "-m", "uv"]
            subprocess.run([*uv, "venv", "-q", "--python", CONFIG["teacher"]["python"], str(venv)],
                           check=True)
            subprocess.run([*uv, "pip", "install", "-q", "--python", str(venv / "bin" / "python"),
                            CONFIG["teacher"]["vllm_spec"], "numpy", "pyarrow"])
            (venv / ".ready").touch()
        python, backend = str(venv / "bin" / "python"), "vllm"
    gpus = gpu_count()
    shards_file = QWEN / "shards.json"  # fixed at the first session so a resume keeps the split
    if not shards_file.exists():
        json_dump(shards_file, {"num_shards": 2 if SMOKE else max(gpus, 1)})
    n_shards = json.loads(shards_file.read_text())["num_shards"]
    print(f"GPUs: {gpus} | Qwen shards: {n_shards}")

    def label_shard(k):
        env = dict(os.environ, VLLM_CACHE_ROOT=f"/tmp/vllm_cache_{k}")
        if gpus:
            env["CUDA_VISIBLE_DEVICES"] = str(k % gpus)
        cmd = [python, str(script), "--config", str(run_cfg_path), "--mode", "label",
               "--backend", backend, "--deadline", str(deadline), "--shard", str(k),
               "--num-shards", str(n_shards)]
        return subprocess.run(cmd, env=env).returncode

    todo = [k for k in range(n_shards) if not shard_complete(k)]
    with ThreadPoolExecutor(max(1, min(len(todo), max(gpus, 1)) if not SMOKE else 2)) as ex:
        codes = dict(zip(todo, ex.map(label_shard, todo)))
    print("exit codes:", codes)
    if any(c == 3 for c in codes.values()):
        print("vLLM could not start: Qwen labels are missing; see the log above.")
    if all(shard_complete(k) for k in range(n_shards)):
        for mode in ("merge", "gate"):  # pyarrow only, no ML framework
            subprocess.run([sys.executable, str(script), "--config", str(run_cfg_path),
                            "--mode", mode, "--num-shards", str(n_shards)], check=True)
        QWEN_STATUS = "complete"
    else:
        QWEN_STATUS = "incomplete (rerun with this output attached to resume)"
GATE = json.loads((QWEN / "gate.json").read_text()) if (QWEN / "gate.json").exists() else None
print("Qwen:", QWEN_STATUS, "| gate:", GATE)

## 2. RoBERTa-Large teacher and DistilBERT baseline (JAX from here on)
Consecutive sentences are packed into windows of at most 512 tokens and 48 sentences
(`<bos> s1 <sep> s2 <sep> …`); every sentence appears once. Model: KerasHub backbone → mean of
each sentence's token states → `Dense(1)`; BCE on the gold importance; 2 epochs, best
validation kept. Tokenizers come from the `tokenizers` library (same vocabularies as the
presets, checked on load; KerasHub's own tokenizers would need TensorFlow).

In [ ]:
from __future__ import annotations

import gc

import jax
import keras
from keras import ops

print("keras", keras.__version__, "| jax", jax.__version__, "| devices:", jax.devices())
if ON_KAGGLE and not SMOKE:
    assert jax.devices()[0].platform == "gpu", "No GPU: Settings -> Accelerator -> GPU T4 x2."


NO_DECAY = ["bias", "gamma", "beta", "position_embedding"]


@keras.saving.register_keras_serializable(package="dialogue_models")
class LayerwiseAdamW(keras.optimizers.AdamW):
    """AdamW whose learning rate (and decoupled weight decay) is scaled per layer.

    ``lr_multipliers`` maps a layer name to a factor: a variable whose path contains that
    name as one segment (``.../encoder/...``) uses ``learning_rate * factor``. Other variables
    use the base learning rate.
    """

    def __init__(self, lr_multipliers: dict[str, float] | None = None, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.lr_multipliers = dict(lr_multipliers or {})

    def _multiplier(self, variable) -> float:
        segments = variable.path.split("/")
        for name, factor in self.lr_multipliers.items():
            if name in segments:
                return factor
        return 1.0

    def update_step(self, gradient, variable, learning_rate):
        return super().update_step(gradient, variable, learning_rate * self._multiplier(variable))

    def _apply_weight_decay(self, variables) -> None:
        if self.weight_decay is None:
            return
        for variable in variables:
            if self._use_weight_decay(variable):
                lr = keras.ops.cast(self.learning_rate, variable.dtype) * self._multiplier(variable)
                wd = keras.ops.cast(self.weight_decay, variable.dtype)
                variable.assign(variable - variable * wd * lr)

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), "lr_multipliers": self.lr_multipliers}


def build_optimizer(opt_cfg: dict[str, Any], total_steps: int) -> keras.optimizers.Optimizer:
    """AdamW with linear warmup to the peak learning rate, then cosine decay to 0.

    The peak is ``peak_lr`` for every weight, or ``lr_heads`` with ``lr_encoder`` for the
    weights of the layer named ``encoder`` (students: smaller steps for pretrained weights).
    ``warmup_steps`` (absolute) or ``warmup_fraction`` (of ``total_steps``) sets the warmup.
    """
    if opt_cfg["name"] != "adamw" or opt_cfg["schedule"] != "cosine":
        raise ValueError("Only adamw with a cosine schedule is implemented.")
    if "warmup_steps" in opt_cfg:
        warmup = int(opt_cfg["warmup_steps"])
    else:
        warmup = int(opt_cfg["warmup_fraction"] * total_steps)
    warmup = min(warmup, max(total_steps - 1, 0))
    if "lr_heads" in opt_cfg:
        peak = opt_cfg["lr_heads"]
        multipliers = {"encoder": opt_cfg["lr_encoder"] / peak}
    else:
        peak, multipliers = opt_cfg["peak_lr"], {}
    schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0,
        decay_steps=max(total_steps - warmup, 1),
        warmup_target=peak,
        warmup_steps=warmup,
    )
    optimizer = LayerwiseAdamW(
        lr_multipliers=multipliers, learning_rate=schedule, weight_decay=opt_cfg["weight_decay"]
    )
    optimizer.exclude_from_weight_decay(var_names=NO_DECAY)
    return optimizer


def configure_devices(
    mixed_precision: str, *batch_sizes: int, data_parallel: bool = True
) -> dict[str, Any]:
    """Set the dtype policy and, with several GPUs and ``data_parallel``, data parallelism.
    Returns device info.

    With data parallelism every ``batch_sizes`` value must divide evenly across the devices.
    ``data_parallel=False`` uses one GPU: no cross-GPU communication (NCCL failed on the 2x T4
    image of 2026-10, ``Cuda failure 500 'named symbol not found'``).
    """
    keras.mixed_precision.set_dtype_policy(mixed_precision)
    devices = jax.devices() if data_parallel else jax.devices()[:1]
    for bs in batch_sizes:
        if bs % len(devices):
            raise ValueError(f"batch size {bs} is not divisible by the {len(devices)} devices")
    if len(devices) > 1:
        keras.distribution.set_distribution(keras.distribution.DataParallel())
    return {
        "devices": [str(d) for d in devices],
        "platform": devices[0].platform,
        "mixed_precision": mixed_precision,
        "keras": keras.__version__,
        "jax": jax.__version__,
    }


COLUMNS = (
    "note_id",
    "source",
    "sentence_id",
    "speaker",
    "importance",
    "confidence",
    "zone",
    "hard_negative",
    "spans",
)


def layer_of(source: str) -> str:
    """Release layer holding a source: ``synthetic`` (incl. hard negatives) or ``ami``."""
    return "ami" if source == "ami" else "synthetic"


def corpus_split_paths(corpus_dir: str | Path, split: str, sources: Sequence[str]) -> list[Path]:
    """Sentence tables of ``split`` for the layers of ``sources`` (synthetic first, then AMI)."""
    unknown = set(sources) - set(SOURCES)
    if unknown:
        raise ValueError(f"Unknown sources {sorted(unknown)}; expected a subset of {SOURCES}")
    layers = sorted({layer_of(s) for s in sources}, key=("synthetic", "ami").index)
    paths = [
        Path(corpus_dir) / "data" / layer / "sentences" / f"{split}.parquet" for layer in layers
    ]
    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Corpus split not found: {', '.join(map(str, missing))}")
    return paths


def read_corpus_split(
    corpus_dir: str | Path, split: str, text_field: str, sources: Sequence[str]
) -> dict[str, list[dict[str, Any]]]:
    """Rows of one split grouped by meeting, sentences sorted by ``sentence_id``.

    Meetings keep the order of their first row (synthetic layer first). Rows whose ``source``
    is not in ``sources`` are dropped.
    """
    meetings: dict[str, list[dict[str, Any]]] = {}
    for path in corpus_split_paths(corpus_dir, split, sources):
        for row in pq.read_table(path, columns=[*COLUMNS, text_field]).to_pylist():
            if row["source"] in sources:
                row["text"] = row.pop(text_field)
                meetings.setdefault(row["note_id"], []).append(row)
    for rows in meetings.values():
        rows.sort(key=lambda r: r["sentence_id"])
    return meetings


Encode = Callable[[str], list[int]]


@dataclass
class Specials:
    """Special token IDs of a teacher tokenizer."""

    bos: int
    sep: int
    pad: int


@dataclass
class Packed:
    """Packed windows: model inputs, labels and the sentence key of every label slot."""

    token_ids: np.ndarray  # (W, L) int32
    padding_mask: np.ndarray  # (W, L) bool
    sentence_index: np.ndarray  # (W, L) int32, -1 = no sentence
    labels: np.ndarray  # (W, K) float32, -1 = empty slot
    keys: list[list[tuple[str, int]]]  # per window: (note_id, sentence_id) per slot


def pack_meetings(
    meetings: dict[str, list[dict[str, Any]]],
    encode: Encode,
    specials: Specials,
    max_tokens: int,
    max_sentence_tokens: int,
    max_sentences: int,
) -> Packed:
    """Pack every meeting's sentences into windows (see module docstring).

    ``meetings`` maps a note ID to its rows (``sentence_id``, ``text``, ``importance``) in
    meeting order, as :func:`~src.model1_notes.data.read_corpus_split` returns them.
    """
    windows: list[tuple[list[int], list[int], list[float], list[tuple[str, int]]]] = []
    for note_id, rows in meetings.items():
        tokens, index, labels, keys = [specials.bos], [-1], [], []
        for row in rows:
            piece = encode(row["text"])[:max_sentence_tokens] or [specials.sep]
            full = len(tokens) + len(piece) + 1 > max_tokens or len(labels) == max_sentences
            if full and labels:
                windows.append((tokens, index, labels, keys))
                tokens, index, labels, keys = [specials.bos], [-1], [], []
            k = len(labels)
            tokens += [*piece, specials.sep]
            index += [k] * len(piece) + [-1]
            labels.append(float(row["importance"]))
            keys.append((note_id, int(row["sentence_id"])))
        if labels:
            windows.append((tokens, index, labels, keys))
    n = len(windows)
    token_ids = np.full((n, max_tokens), specials.pad, dtype=np.int32)
    sentence_index = np.full((n, max_tokens), -1, dtype=np.int32)
    label_arr = np.full((n, max_sentences), -1.0, dtype=np.float32)
    for w, (tokens, index, labels, _) in enumerate(windows):
        token_ids[w, : len(tokens)] = tokens
        sentence_index[w, : len(index)] = index
        label_arr[w, : len(labels)] = labels
    mask = np.zeros((n, max_tokens), dtype=bool)
    for w, (tokens, *_rest) in enumerate(windows):
        mask[w, : len(tokens)] = True
    return Packed(token_ids, mask, sentence_index, label_arr, [w[3] for w in windows])


@keras.saving.register_keras_serializable(package="dialogue_models")
class SentenceMeanPool(keras.layers.Layer):
    """``(hidden (B, L, d), sentence_index (B, L)) -> (B, K, d)``: mean token state per sentence."""

    def __init__(self, max_sentences: int, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.max_sentences = max_sentences

    def call(self, hidden, sentence_index):
        onehot = ops.one_hot(ops.cast(sentence_index, "int32"), self.max_sentences)  # -1 -> 0s
        onehot = ops.cast(onehot, hidden.dtype)
        summed = ops.einsum("blk,bld->bkd", onehot, hidden)
        counts = ops.maximum(ops.sum(onehot, axis=1), 1.0)[:, :, None]
        return summed / counts

    def compute_output_shape(self, hidden_shape, sentence_index_shape=None):
        return (hidden_shape[0], self.max_sentences, hidden_shape[-1])

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), "max_sentences": self.max_sentences}


@keras.saving.register_keras_serializable(package="dialogue_models")
class SentenceBCE(keras.losses.Loss):
    """Binary cross-entropy on logits per sentence slot (empty slots weighted 0)."""

    def __init__(
        self, name: str = "sentence_bce", reduction: str = "mean_with_sample_weight", **kwargs: Any
    ) -> None:
        super().__init__(name=name, reduction=reduction, **kwargs)

    def call(self, y_true, y_pred):
        return ops.binary_crossentropy(
            ops.cast(y_true, "float32"), ops.cast(y_pred, "float32"), from_logits=True
        )


def build_sentence_classifier(
    backbone: keras.Model, max_tokens: int, max_sentences: int
) -> keras.Model:
    """Backbone + per-sentence mean pooling + ``Dense(1)`` logit per sentence."""
    token_ids = keras.Input((max_tokens,), dtype="int32", name="token_ids")
    padding_mask = keras.Input((max_tokens,), dtype="bool", name="padding_mask")
    sentence_index = keras.Input((max_tokens,), dtype="int32", name="sentence_index")
    hidden = backbone({"token_ids": token_ids, "padding_mask": padding_mask})
    pooled = SentenceMeanPool(max_sentences, name="sentence_pool")(hidden, sentence_index)
    logits = keras.layers.Dense(1, dtype="float32", name="sentence_logit")(pooled)
    logits = keras.layers.Reshape((max_sentences,), dtype="float32", name="logits")(logits)
    inputs = {
        "token_ids": token_ids,
        "padding_mask": padding_mask,
        "sentence_index": sentence_index,
    }
    return keras.Model(inputs, logits, name="sentence_classifier")


def inputs_of(packed: Packed) -> dict[str, np.ndarray]:
    return {
        "token_ids": packed.token_ids,
        "padding_mask": packed.padding_mask,
        "sentence_index": packed.sentence_index,
    }


def fine_tune(
    model: keras.Model,
    train: Packed,
    validation: Packed,
    cfg: dict[str, Any],
    seed: int,
) -> keras.callbacks.History:
    """Fine-tune on gold importance (BCE, empty slots weighted 0), keep the best validation."""
    keras.utils.set_random_seed(seed)
    steps = max(1, len(train.labels) // cfg["batch_size"]) * cfg["epochs"]
    optimizer = build_optimizer(
        {
            "name": "adamw",
            "schedule": "cosine",
            "peak_lr": cfg["peak_lr"],
            "warmup_fraction": cfg["warmup_fraction"],
            "weight_decay": cfg["weight_decay"],
        },
        steps,
    )
    model.compile(optimizer=optimizer, loss=SentenceBCE())

    def weights(p: Packed) -> np.ndarray:
        return (p.labels >= 0).astype(np.float32)

    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
    return model.fit(
        inputs_of(train),
        np.clip(train.labels, 0, 1),
        sample_weight=weights(train),
        validation_data=(
            inputs_of(validation),
            np.clip(validation.labels, 0, 1),
            weights(validation),
        ),
        batch_size=cfg["batch_size"],
        epochs=cfg["epochs"],
        callbacks=[stop],
        verbose=2,
    )


def predict_logits(model: keras.Model, packed: Packed, batch_size: int) -> list[dict[str, Any]]:
    """One row per sentence: ``note_id``, ``sentence_id``, ``logit``."""
    logits = np.asarray(model.predict(inputs_of(packed), batch_size=batch_size, verbose=0))
    rows = []
    for w, keys in enumerate(packed.keys):
        for k, (note_id, sentence_id) in enumerate(keys):
            rows.append(
                {"note_id": note_id, "sentence_id": sentence_id, "logit": float(logits[w, k])}
            )
    return rows


def write_logits(path: str | Path, rows: Sequence[dict[str, Any]]) -> None:
    """Teacher table read by the student (``train.align_teacher``) and the evaluation."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    schema = pa.schema(
        [("note_id", pa.string()), ("sentence_id", pa.int32()), ("logit", pa.float32())]
    )
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(pa.Table.from_pylist(list(rows), schema=schema), path)


def hf_tokenizer(name: str, specials: dict[str, str]) -> tuple[Encode, Specials, int]:
    """``encode``, special IDs and vocabulary size of a Hugging Face tokenizer (teacher only)."""
    from tokenizers import Tokenizer

    tok = Tokenizer.from_pretrained(name)

    def encode(text: str) -> list[int]:
        return tok.encode(text, add_special_tokens=False).ids

    ids = {k: tok.token_to_id(v) for k, v in specials.items()}
    if None in ids.values():
        raise ValueError(f"{name}: missing special tokens {specials}")
    return encode, Specials(**ids), tok.get_vocab_size()


def find_local_preset(root: str | Path, preset: str) -> Path | None:
    """An attached KerasHub preset under ``root``: a directory holding ``config.json`` whose
    path contains ``preset`` (Kaggle mounts ``keras/<model>/keras/<preset>/<version>/``)."""
    root = Path(root)
    if not root.exists():
        return None
    found = sorted(
        (Path(d) for d, _, files in os.walk(root, followlinks=True) if "config.json" in files),
        key=lambda p: (preset not in p.parts, len(p.parts)),
    )
    return next((p for p in found if preset in p.parts), None)


def keras_hub_backbone(
    preset: str, vocab_size: int, search_root: str | Path | None = None
) -> keras.Model:  # pragma: no cover - Kaggle only
    """Pretrained KerasHub backbone; checks the vocabulary size.

    A non-interactive Kaggle session cannot attach models while it runs, so the preset must be
    attached as an input (Add Input -> Models); it is loaded from that local copy.
    """
    import keras_hub

    local = None if search_root is None else find_local_preset(search_root, preset)
    if search_root is not None and local is None:
        raise SystemExit(
            f"Attach the KerasHub model '{preset}' (Add Input -> Models -> Keras) and rerun."
        )
    backbone = keras_hub.models.Backbone.from_preset(str(local) if local else preset)
    if backbone.vocabulary_size != vocab_size:
        raise ValueError(
            f"{preset}: vocabulary {backbone.vocabulary_size} != tokenizer {vocab_size}"
        )
    return backbone


def run_encoder(
    name: str,
    backbone: keras.Model,
    encode: Encode,
    specials: Specials,
    cfg: dict[str, Any],
    corpus_dir: str | Path,
    out_dir: str | Path,
    seed: int,
    sources: Sequence[str] = ("synthetic", "synthetic_hn", "ami"),
) -> dict[str, Any]:
    """Fine-tune ``backbone`` on train, then write ``<out_dir>/<name>/<split>.parquet`` logits
    for every split. Returns the training history and counts."""

    def pack(split: str) -> Packed:
        meetings = read_corpus_split(corpus_dir, split, "sentence_text", sources)
        return pack_meetings(
            meetings,
            encode,
            specials,
            cfg["max_tokens"],
            cfg["max_sentence_tokens"],
            cfg["max_sentences"],
        )

    packed = {split: pack(split) for split in ("train", "validation", "test")}
    model = build_sentence_classifier(backbone, cfg["max_tokens"], cfg["max_sentences"])
    history = fine_tune(model, packed["train"], packed["validation"], cfg, seed)
    counts = {}
    for split, p in packed.items():
        rows = predict_logits(model, p, cfg["batch_size"])
        write_logits(Path(out_dir) / name / f"{split}.parquet", rows)
        counts[split] = len(rows)
    return {
        "history": {k: [float(x) for x in v] for k, v in history.history.items()},
        "sentences": counts,
    }

In [ ]:
def toy_encoder(max_tokens):
    """Smoke runs only: a tiny random encoder and a toy tokenizer (nothing downloaded)."""
    vocab = {}

    def encode(text):
        return [4 + vocab.setdefault(w, len(vocab) % 90) for w in text.lower().split()]

    token_ids = keras.Input((max_tokens,), dtype="int32", name="token_ids")
    padding_mask = keras.Input((max_tokens,), dtype="bool", name="padding_mask")
    hidden = keras.layers.Dense(16)(keras.layers.Embedding(100, 16)(token_ids))
    backbone = keras.Model({"token_ids": token_ids, "padding_mask": padding_mask}, hidden)
    return backbone, encode, Specials(bos=1, sep=2, pad=0)


ENC = CONFIG["encoders"]
ENCODERS = {}
configure_devices(ENC["mixed_precision"], ENC["batch_size"], data_parallel=ENC["data_parallel"])
for name in ("roberta", "distilbert"):
    done = all((LABELS / name / f"{s}.parquet").exists() for s in ("train", "validation", "test"))
    if done:
        ENCODERS[name] = "complete (restored)"
        continue
    if hours_left() < 1.0 and not SMOKE:
        ENCODERS[name] = "skipped: not enough time left (rerun to resume)"
        continue
    start = time.time()
    if SMOKE:
        backbone, encode, specials = toy_encoder(ENC["max_tokens"])
    else:
        spec = ENC[name]
        encode, specials, vocab_size = hf_tokenizer(spec["hf_tokenizer"], spec["specials"])
        backbone = keras_hub_backbone(spec["preset"], vocab_size, INPUT)
    result = run_encoder(name, backbone, encode, specials, ENC, CORPUS, LABELS, CONFIG["seed"])
    ENCODERS[name] = {"status": "complete", "seconds": round(time.time() - start, 1), **result}
    print(name, ENCODERS[name])
    del backbone
    keras.backend.clear_session()
    gc.collect()

## 3. Manifest

In [ ]:
stats = {}
for k in range(json.loads((QWEN / "shards.json").read_text())["num_shards"]
               if (QWEN / "shards.json").exists() else 0):
    path = QWEN / "shards" / str(k) / "stats.json"
    if path.exists():
        stats[str(k)] = json.loads(path.read_text())
json_dump(WORK / f"manifest_{STAGE}.json", {
    "stage": STAGE,
    "smoke": SMOKE,
    "config": CONFIG,
    "corpus": str(CORPUS),
    "qwen": {"status": QWEN_STATUS, "gate": GATE, "shards": stats},
    "encoders": ENCODERS,
    "hours": round((time.time() - SESSION_START) / 3600, 3),
})
print(json.dumps({"qwen": QWEN_STATUS, "gate": GATE,
                  "encoders": {k: v if isinstance(v, str) else v["status"]
                               for k, v in ENCODERS.items()}}, indent=2))